# FinTok AI · 04_video / 01 · Montador de vídeo (demo)

**Objetivo:** convertir un guion en JSON (texto + audio por escena) en un vídeo vertical 1080×1920 para redes sociales.

**Dos modos de avatar:**
- **Rápido (gratis, sin GPU):** avatar con 3 posiciones de boca que siguen el volumen de la voz, con rebote y balanceo.
- **Realista:** vídeo de la cara hablando generado con **EchoMimic** (Space público de Hugging Face, vía `gradio_client`). Si el Space falla, cada escena usa automáticamente el modo rápido.

**Archivos que hay que subir:** `compositor.py`, `avatar_echomimic.py` y los avatares `andrea_closed.png`, `andrea_half.png`, `andrea_open.png`.

**Para el modo realista:** un token de Hugging Face guardado como secreto `HF_TOKEN` (icono 🔑).

## 0. Instalación

Los avisos de pip sobre versiones de `click` son normales y no afectan.

In [ ]:
!pip install -q gTTS gradio_client

## 1. Subir archivos

Ejecuta la celda y sube `compositor.py`, `avatar_echomimic.py` y los tres PNG del avatar.
Si están en carpetas distintas, ejecuta la celda dos veces (una para los `.py` y otra para los PNG).

In [ ]:
import os, shutil, json
from google.colab import files

uploaded = files.upload()
os.makedirs("assets/avatar", exist_ok=True)
for name in uploaded:
    if name.endswith(".png"):
        shutil.move(name, f"assets/avatar/{name}")
print("Avatares:", sorted(os.listdir("assets/avatar")))
print("Código:", [f for f in os.listdir(".") if f.endswith(".py")])

## 2. Importar los módulos

In [ ]:
try:  # dentro del repo
    from src.video.compositor import render_video, Style
    from src.video.avatar_echomimic import attach_talking_heads
except ModuleNotFoundError:  # .py subidos sueltos (Colab)
    from compositor import render_video, Style
    from avatar_echomimic import attach_talking_heads

from IPython.display import Video

## 3. Elegir el guion

Ejecuta **solo una** de las dos opciones.

**Opción A:** guion de prueba (cifras de relleno).

In [ ]:
spec = {
    "company": "NVIDIA", "year": 2025, "language": "es",
    "scenes": [
        {"scene_id": 1, "scene_type": "hook", "speaker": "andrea",
         "text": "NVIDIA ha más que duplicado sus ingresos en un solo año.",
         "key_figure": "x2", "source": "NVIDIA 10-K FY2025", "verified": True},
        {"scene_id": 2, "scene_type": "dato", "speaker": "andrea",
         "text": "El negocio de data center ya supone la gran mayoría de las ventas.",
         "key_figure": "~88%", "source": "NVIDIA 10-K FY2025", "verified": True},
        {"scene_id": 3, "scene_type": "cierre", "speaker": "andrea",
         "text": "¿El riesgo? Una fuerte concentración en pocos clientes.",
         "key_figure": None, "source": "NVIDIA 10-K FY2025", "verified": True},
    ],
}

**Opción B:** JSON real generado por la parte de texto (sube el archivo `.json`).

In [ ]:
uploaded = files.upload()
JSON_PATH = list(uploaded.keys())[0]
spec = json.load(open(JSON_PATH, encoding="utf-8"))
print(spec["company"], spec["year"], "·", len(spec["scenes"]), "escenas")

## 4. Voces de prueba

Mientras no esté la voz real (TTS de la parte de audio), generamos audios con gTTS en el idioma del JSON.
Las escenas que ya traen `audio_path` no se tocan.

In [ ]:
from gtts import gTTS

os.makedirs("mock", exist_ok=True)
for s in spec["scenes"]:
    if not s.get("audio_path"):
        mp3 = f"mock/{spec['company']}_{s['scene_id']:02d}.mp3"
        wav = mp3.replace(".mp3", ".wav")
        gTTS(s["text"], lang=spec.get("language", "es")).save(mp3)
        os.system(f"ffmpeg -y -loglevel error -i {mp3} -ar 24000 -ac 1 {wav}")
        s["audio_path"] = wav
print([s["audio_path"] for s in spec["scenes"]])

## 5. Vídeo con el avatar rápido (3 bocas)

Tarda unos segundos por cada segundo de vídeo.

In [ ]:
out_rapido = render_video(spec, avatar_dir="assets/avatar",
                          out_path=f"outputs/{spec['company']}_{spec['year']}_rapido.mp4")
Video(out_rapido, embed=True, width=360)

## 6. Vídeo con el avatar realista (EchoMimic)

Usa la GPU gratuita del Space, que tiene una **cuota diaria**. Cada escena tarda unos minutos.
Los vídeos se guardan en caché (`outputs/avatar_videos/`): si vuelves a ejecutar, no se regeneran.

**6.1** Prueba con una sola escena corta.

In [ ]:
from google.colab import userdata
HF_TOKEN = userdata.get("HF_TOKEN")

corta = min(spec["scenes"], key=lambda s: len(s["text"]))
spec_prueba = {**spec, "scenes": [dict(corta)]}
spec_prueba = attach_talking_heads(spec_prueba, avatar_dir="assets/avatar", token=HF_TOKEN, steps=20)

out_prueba = render_video(spec_prueba, avatar_dir="assets/avatar", out_path="outputs/prueba_echomimic.mp4")
Video(out_prueba, embed=True, width=360)

**6.2** Si la prueba ha ido bien, todas las escenas.

In [ ]:
spec_real = attach_talking_heads(json.loads(json.dumps(spec)), avatar_dir="assets/avatar",
                                 token=HF_TOKEN, steps=20)
out_real = render_video(spec_real, avatar_dir="assets/avatar",
                        out_path=f"outputs/{spec['company']}_{spec['year']}_echomimic.mp4")
Video(out_real, embed=True, width=360)

## 7. Ajustes de estilo (opcional)

Todo lo visual está en `Style`: colores, tamaño y posición de la burbuja, sensibilidad de la boca, textos legales…

In [ ]:
estilo = Style(accent=(255, 196, 0), bubble_d=620, mouth_thresholds=(0.10, 0.35))
out_estilo = render_video(spec, avatar_dir="assets/avatar", out_path="outputs/demo_estilo.mp4", style=estilo)
Video(out_estilo, embed=True, width=360)

## 8. Descargar los vídeos y los avatares generados

In [ ]:
!zip -qr resultados.zip outputs
files.download("resultados.zip")